In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin
from catboost import CatBoostRegressor

In [ ]:
df_parkings = pd.read_parquet('Dane/parkings.parquet')

In [ ]:
df_parkings_availabilities = pd.read_parquet('Dane/parking_availabilities.parquet')

In [ ]:
df_sks_users = pd.read_parquet('Dane/sks_users.parquet')

In [ ]:
pd.set_option('display.max_columns',30)
pd.set_option('display.max_rows',100)

# Anomally detection and handling 

In [ ]:
df_parkings.head(10)

### Fixing missing values in df_parkings
Polinka parking doesn't have sepcified opinig and closing hours because it's always open. To avoid confiusion decided to manully set opeing and closing times to 00:00:00


In [ ]:
df_parkings.loc[df_parkings['name'] == "Polinka", ['open_hour', 'close_hour']] = "00:00:00"

In [ ]:
df_parkings.head(10)


In [ ]:
df_parkings_availabilities = df_parkings_availabilities.sort_values(by=['measured_at'])
df_parkings_availabilities = df_parkings_availabilities.reset_index(drop=True)
df_parkings_availabilities.head(10)

In [ ]:
df_parkings_availabilities['day'] = df_parkings_availabilities['measured_at'].dt.day_name()

df_parkings_availabilities[['measured_at', 'day']].tail()

In [ ]:
def is_open_parking(row):
    if row['parking_id'] == 2:
        return True
    
    time = row['measured_at'].time()
    if row['parking_id'] == 4:
        # check if between 6:00 and 22:00
        if dt.time(6, 0) <= time < dt.time(22, 0):
            return True
        else:
            return False
    else:
        # check if between 6:00 and 22:30
        if dt.time(6, 0) <= time < dt.time(22, 30):
            return True
        else:
            return False
    

In [ ]:
df_parkings_availabilities['is_open'] = df_parkings_availabilities.apply(is_open_parking, axis=1)

In [ ]:
df_parking_architektura = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 7]
df_parking_architektura = df_parking_architektura.reset_index(drop=True)

df_parking_wronskiego = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 4]
df_parking_wronskiego = df_parking_wronskiego.reset_index(drop=True)

df_parking_polinka = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 2]
df_parking_polinka = df_parking_polinka.reset_index(drop=True)

df_parking_D20 = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 5]
df_parking_D20 = df_parking_D20.reset_index(drop=True)

df_parking_geocentrum = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 6]
df_parking_geocentrum = df_parking_geocentrum.reset_index(drop=True)

In [ ]:
def check_missing_logs(df_to_check, time_column_name, time_between_logs = 60, allowed_treshold = 10):
    df_to_check = df_to_check.sort_values(time_column_name)
    df_to_check['time_since_prev'] = df_to_check[time_column_name].diff()
    
    threshold =  pd.Timedelta(seconds=allowed_treshold+time_between_logs)
    
    gaps = df_to_check[df_to_check['time_since_prev'] > threshold]
    
    print(f"Found {len(gaps)} gaps in data logging.")
    print(gaps[[time_column_name, 'time_since_prev']].sort_values(by=['time_since_prev'],  ascending=False).head(5))

In [ ]:
check_missing_logs(df_parking_architektura, 'measured_at')

In [ ]:
check_missing_logs(df_parking_wronskiego, 'measured_at')

In [ ]:
check_missing_logs(df_parking_polinka, 'measured_at')

In [ ]:
check_missing_logs(df_parking_D20, 'measured_at')

In [ ]:
check_missing_logs(df_parking_geocentrum, 'measured_at')

In [ ]:
agg_rules = {
    'spaces_left': 'mean',
    'trend': 'mean',
    'parking_id': 'first',
    'day': 'first', 
    'is_open': 'max',
}
df_parking_architektura_5min = df_parking_architektura.resample('5min', on='measured_at').agg(agg_rules)
df_parking_wronskiego_5min = df_parking_wronskiego.resample('5min', on='measured_at').agg(agg_rules)
df_parking_polinka_5min = df_parking_polinka.resample('5min', on='measured_at').agg(agg_rules)
df_parking_D20_5min = df_parking_D20.resample('5min', on='measured_at').agg(agg_rules)
df_parking_geocentrum_5min = df_parking_geocentrum.resample('5min', on='measured_at').agg(agg_rules)


In [ ]:
parking_lots = [
    ("Parking Architektura", df_parking_architektura_5min),
    ("Parking Wronskiego", df_parking_wronskiego_5min),
    ("Parking Polinka", df_parking_polinka_5min),
    ("Parking D20", df_parking_D20_5min),
    ("Parking Geocentrum", df_parking_geocentrum_5min)
]

fig = make_subplots(
    rows=len(parking_lots), 
    cols=1, 
    shared_xaxes=True, 
    subplot_titles=[item[0] for item in parking_lots], # item[0] is the title
    vertical_spacing=0.08
)

for i, (title, df) in enumerate(parking_lots):

    dff = df.copy()
    dff["is_open"] = dff["is_open"].astype(str)
    
    temp_fig = px.line(
        dff, 
        x=dff.index, 
        y="spaces_left", 
        color="is_open",
        color_discrete_map={"True": "red", "False": "blue"}
    )
    

    for trace in temp_fig.data:
        trace.legendgroup = trace.name 
        
        if i > 0:
            trace.showlegend = False
            
        fig.add_trace(trace, row=i+1, col=1)


fig.update_layout(
    height=1500,  
    width=1500,
    title_text="Parking Availability Analysis",
    hovermode="x unified"
)

fig.update_xaxes(title_text="Measured At", row=5, col=1)

fig.show()

### Missing data in parkings

There are few gaps, but nothing major; the longest gap is 5 days, which suggests it can be easily imputed using data from the previous week. High correlation between the lag 7 day feature suggests the correctness of this solution. Shorter gaps can be imputed linearly. The recommended approach is that gaps under 1 hour will be imputed using linear inputting, and larger gaps will be taken care of using lag  features.

### Problem of negatvie spaces left and overbooked parking lots

Charts and data suggest it's possible for sensors to report negative spaces left. This is an anomaly and will be sliced to stop at 0 with the added flag paring_is_overbooked. There is also a significant jump in available parking spaces in Wronskiego parking. The new limit seems to correspond with the declared max capacity for this parking, but perhaps there should be a new record added, which could explain the sudden jump from 192 free spaces to 207. 

Jump from 192 to 207 frre spaces actually coresponds to just ~8% so it can be ignorred. 

In [ ]:
df_parking_architektura_5min.head(10)

In [ ]:
class ParkingSpacesInputer(BaseEstimator, TransformerMixin):
    def __init__(self, freq_minutes=5):
        self.freq_minutes = freq_minutes

    def fit(self, X, y=None):
        self.fill_value_ = X['spaces_left'].median()
        return self  

    def transform(self, X):
        X = X.copy()
        if not X.index.is_monotonic_increasing:
            X = X.sort_index()
            
        X['is_imputed'] = X['spaces_left'].isna().astype(int)

        #Linear interpolation for samll gaps up to 1 hour
        limit_small = 60 // self.freq_minutes
        X['spaces_left'] = X['spaces_left'].interpolate(method='linear', limit=limit_small)

        #For larger gaps use value from exactly 7 days ago
        records_per_week = 7 * 24 * (60 // self.freq_minutes)
        val_7d_ago = X['spaces_left'].shift(records_per_week)
        X['spaces_left'] = X['spaces_left'].fillna(val_7d_ago)

        cols_to_fix = X.select_dtypes(include=[np.number]).columns
        cols_to_fix = [c for c in cols_to_fix if c not in ['parking_id', 'spaces_left']]

        for col in cols_to_fix:
            val_7d_ago = X[col].shift(records_per_week)
            X[col] = X[col].fillna(val_7d_ago)
        

        #Parking id stays the same 
        X = X.ffill()
        X = X.fillna(self.fill_value_)
        return X

parking_space_inputer = ParkingSpacesInputer()

In [ ]:
df_parking_architektura = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 7]
df_parking_architektura = df_parking_architektura.reset_index(drop=True)

df_parking_wronskiego = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 4]
df_parking_wronskiego = df_parking_wronskiego.reset_index(drop=True)

df_parking_polinka = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 2]
df_parking_polinka = df_parking_polinka.reset_index(drop=True)

df_parking_D20 = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 5]
df_parking_D20 = df_parking_D20.reset_index(drop=True)

df_parking_geocentrum = df_parkings_availabilities[df_parkings_availabilities['parking_id'] == 6]
df_parking_geocentrum = df_parking_geocentrum.reset_index(drop=True)

In [ ]:
agg_rules = {
    'spaces_left': 'mean',
    # 'trend': 'mean',
    'parking_id': 'first',
    'day': 'first', 
    'is_open': 'max',
}
df_parking_architektura_5min = df_parking_architektura.resample('5min', on='measured_at').agg(agg_rules)
df_parking_wronskiego_5min = df_parking_wronskiego.resample('5min', on='measured_at').agg(agg_rules)
df_parking_polinka_5min = df_parking_polinka.resample('5min', on='measured_at').agg(agg_rules)
df_parking_D20_5min = df_parking_D20.resample('5min', on='measured_at').agg(agg_rules)
df_parking_geocentrum_5min = df_parking_geocentrum.resample('5min', on='measured_at').agg(agg_rules)


In [ ]:
df_parking_architektura_5min_inputed = parking_space_inputer.fit_transform(df_parking_architektura_5min)
df_parking_wronskiego_5min_inputed = parking_space_inputer.fit_transform(df_parking_wronskiego_5min)
df_parking_polinka_5min_inputed = parking_space_inputer.fit_transform(df_parking_polinka_5min)
df_parking_D20_5min_inputed = parking_space_inputer.fit_transform(df_parking_D20_5min)
df_parking_geocentrum_5min_inputed = parking_space_inputer.fit_transform(df_parking_geocentrum_5min)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(50, 50), sharex=True)

sns.Sca(data=df_parking_architektura_5min_inputed, x="measured_at", y="spaces_left", hue="is_imputed", ax=axes[0])
axes[0].set_title("Parking architektura is_imputed")

sns.lineplot(data=df_parking_wronskiego_5min_inputed, x="measured_at", y="spaces_left", hue="is_imputed", ax=axes[1])
axes[1].set_title("Parking wronskiego is_imputed")

sns.lineplot(data=df_parking_polinka_5min_inputed, x="measured_at", y="spaces_left", hue="is_imputed", ax=axes[2])
axes[2].set_title("Parking polinka is_imputed")

sns.lineplot(data=df_parking_D20_5min_inputed, x="measured_at", y="spaces_left", hue="is_imputed", ax=axes[3])
axes[3].set_title("Parking D20 is_imputed")

sns.lineplot(data=df_parking_geocentrum_5min_inputed, x="measured_at", y="spaces_left", hue="is_imputed", ax=axes[4])
axes[4].set_title("Parking geocentrum is_imputed")

plt.show()

In [ ]:
print("Parking Architektura:")
print(df_parking_architektura_5min_inputed.isna().sum())
print("Parking Wronskiego:")
print(df_parking_wronskiego_5min_inputed.isna().sum())
print("Parking Polinka:")
print(df_parking_polinka_5min_inputed.isna().sum())
print("Parking D20:")
print(df_parking_D20_5min_inputed.isna().sum())
print("Parking Geocentrum:")
print(df_parking_geocentrum_5min_inputed.isna().sum())

Charts suggest that data was inputed correctly. Some more testing might be needed but at first glance data looks very good after inputing. There is one issuie worth noting i use only data from the past when inputing something so if we are missing day in week with high activity it might get inputed with a day with lower activity. is_inputed flag should tell model celarly what to do. 

In [ ]:
time_cols = ['created_at', 'measured_at', 'updated_at']

time_spread = df_parkings_availabilities[time_cols].max(axis=1) - df_parkings_availabilities[time_cols].min(axis=1)

anomalies = df_parkings_availabilities[time_spread > pd.Timedelta(seconds=60)]
print(f"Found {len(anomalies)} anomalies where time gap > 60s.")
anomalies.head(20)

In [ ]:
time_cols = ['created_at', 'measured_at', 'updated_at']

time_spread = df_parkings_availabilities[time_cols].max(axis=1) - df_parkings_availabilities[time_cols].min(axis=1)

anomalies = df_parkings_availabilities[time_spread > pd.Timedelta(seconds=240)]
print(f"Found {len(anomalies)} anomalies where time gap > 240s.")
anomalies.head(20)

In [ ]:
df_parkings_availabilities.iloc[1787440:1787460]

In [ ]:
gap_check = df_parking_architektura_5min_inputed.loc['2025-11-28 23:50:00':'2025-12-01 08:40:00']
print(gap_check[gap_check['is_imputed'] == 1])

Time gap of over 60 seconds exists for over 779 records, but a time gap of over 240 seconds exists only for 5 records (once for every parking id) This indicates a logging crash, which was later restored and allowed this data to be saved 3 days later. Other gaps (less than 4 minutes) can be ignored as they are statistically insignificant and will be fixed by resampling data to 5 minutes. This logging delay of over 3 days is a significant anomaly and corresponds to 3 days of missing data. But as can be seen above, resampling and inputting the data has automatically filled this gap. 

In [ ]:
expansions = {
    "Parking Wrońskiego": '2025-05-21',
    #Others are ommited so they will be treated as never expanded (add them if they were expanded)
}
size_before_expansion ={
    "Parking Wrońskiego": 192,
    #Others are ommited so they will be treated as never expanded (add them if they were expanded)
}

In [ ]:
df_parking_architektura_5min_inputed.head()

In [ ]:
class ParkingSpacesCorrector(BaseEstimator, TransformerMixin):
    def __init__(self, parkings_df, expansion_date_map, previous_size_map):
        self.parkings_df = parkings_df
        self.expansion_date_map = expansion_date_map
        self.previous_size_map = previous_size_map

    def fit(self, X, y=None):
        return self  

    def transform(self, X):
        X = X.copy()
        # Set the flag
        X['overbooking_detected'] = (X['spaces_left'] < 0).astype(int)
        X['spaces_left'] = X['spaces_left'].clip(lower=0)

        parking_id = X['parking_id'].iloc[0]
        parking_data = self.parkings_df[self.parkings_df['id'] == parking_id].iloc[0]
        parking_name = parking_data['name']
        # Get date from dict; if not there, use a date far in the future
        exp_date_str = self.expansion_date_map.get(parking_name, "2200-01-01")
        exp_date = pd.to_datetime(exp_date_str).tz_localize('UTC')
        X['is_expanded'] = (X.index >= pd.to_datetime(exp_date)).astype(int)

        cap_after =parking_data['places']
        cap_before = self.previous_size_map.get(parking_name, cap_after)
        X['total_capacity'] = np.where(X['is_expanded'] == 1, cap_after, cap_before)
        X['utilization_rate'] = (X['total_capacity'] - X['spaces_left']) / X['total_capacity']
        return X

parking_spaces_corrector = ParkingSpacesCorrector(df_parkings, expansions, size_before_expansion)

In [ ]:
df_parking_architektura_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_architektura_5min_inputed)
df_parking_wronskiego_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_wronskiego_5min_inputed)
df_parking_polinka_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_polinka_5min_inputed)
df_parking_D20_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_D20_5min_inputed)
df_parking_geocentrum_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_geocentrum_5min_inputed)
print(df_parking_architektura_5min_clipped['overbooking_detected'].value_counts())
print(df_parking_wronskiego_5min_clipped['overbooking_detected'].value_counts())
print(df_parking_polinka_5min_clipped['overbooking_detected'].value_counts())
print(df_parking_D20_5min_clipped['overbooking_detected'].value_counts())
print(df_parking_geocentrum_5min_clipped['overbooking_detected'].value_counts())

print("=" * 30)
print(df_parking_architektura_5min_clipped['is_expanded'].value_counts())
print(df_parking_wronskiego_5min_clipped['is_expanded'].value_counts())
print(df_parking_polinka_5min_clipped['is_expanded'].value_counts())
print(df_parking_D20_5min_clipped['is_expanded'].value_counts())
print(df_parking_geocentrum_5min_clipped['is_expanded'].value_counts())

In [ ]:
parking_lots = [
    ("Parking Architektura", df_parking_architektura_5min_clipped),
    ("Parking Wrońskiego", df_parking_wronskiego_5min_clipped), 
    ("Parking Polinka", df_parking_polinka_5min_clipped),
    ("Parking D20", df_parking_D20_5min_clipped),
    ("Parking Geocentrum", df_parking_geocentrum_5min_clipped)
]

fig = make_subplots(
    rows=len(parking_lots), 
    cols=1, 
    shared_xaxes=True, 
    subplot_titles=[item[0] for item in parking_lots],
    vertical_spacing=0.05 
)

for i, (title, df) in enumerate(parking_lots):
    
    temp_fig = px.line(
        df, 
        x=df.index, 
        y="utilization_rate",
        labels={"utilization_rate": "Utilization (%)"}
    )
    
    for trace in temp_fig.data:
        trace.line.color = "royalblue"
        trace.showlegend = False 
        fig.add_trace(trace, row=i+1, col=1)


fig.update_layout(
    height=1200, 
    width=1400,
    title_text="Parking Utilization Rate Over Time",
    hovermode="x unified",
    template="plotly_white" 
)
fig.update_yaxes(range=[0, 1.1], tickformat=".0%")

fig.update_xaxes(title_text="Measured At", row=len(parking_lots), col=1)

fig.show()

Fixed one anomaly only to discover 2 more parkings: "D20" and "Geocentrum" always have some utilization, which might suggest that the sensor is stuck. It probably registers "ghost" cars; perhaps some cars manage to drive out without punching out.

In [ ]:
target_id = df_parking_D20_5min_clipped['parking_id'].iloc[0]
declared_capacity = df_parkings[df_parkings['id'] == target_id]['places'].iloc[0]
max_spaces_left = df_parking_D20_5min_clipped['spaces_left'].max()

print(f"Max free spaces on D20 parking: {max_spaces_left}")
print(f"Declared capacity of D20 parking: {declared_capacity}")
print(f"Difference: {declared_capacity - max_spaces_left}")

In [ ]:
target_id = df_parking_geocentrum_5min_clipped['parking_id'].iloc[0]
declared_capacity = df_parkings[df_parkings['id'] == target_id]['places'].iloc[0]
max_spaces_left = df_parking_geocentrum_5min_clipped['spaces_left'].max()

print(f"Max free spaces on Geocentrum parking: {max_spaces_left}")
print(f"Declared capacity of Geocentrum parking: {declared_capacity}")
print(f"Difference: {declared_capacity - max_spaces_left}")

Correct course of action seem to be changing values in df_parkings to factual ones. 

In [ ]:
df_parkings.loc[df_parkings['id'] == 5, 'places'] = 49
df_parkings.loc[df_parkings['id'] == 6, 'places'] = 267

In [ ]:
df_parkings.head()

In [ ]:
parking_spaces_corrector = ParkingSpacesCorrector(df_parkings, expansions, size_before_expansion)
df_parking_architektura_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_architektura_5min_inputed)
df_parking_wronskiego_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_wronskiego_5min_inputed)
df_parking_polinka_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_polinka_5min_inputed)
df_parking_D20_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_D20_5min_inputed)
df_parking_geocentrum_5min_clipped = parking_spaces_corrector.fit_transform(df_parking_geocentrum_5min_inputed)

In [ ]:
parking_lots = [
    ("Parking Architektura", df_parking_architektura_5min_clipped),
    ("Parking Wrońskiego", df_parking_wronskiego_5min_clipped), 
    ("Parking Polinka", df_parking_polinka_5min_clipped),
    ("Parking D20", df_parking_D20_5min_clipped),
    ("Parking Geocentrum", df_parking_geocentrum_5min_clipped)
]

fig = make_subplots(
    rows=len(parking_lots), 
    cols=1, 
    shared_xaxes=True, 
    subplot_titles=[item[0] for item in parking_lots],
    vertical_spacing=0.05 
)

for i, (title, df) in enumerate(parking_lots):
    
    temp_fig = px.line(
        df, 
        x=df.index, 
        y="utilization_rate",
        labels={"utilization_rate": "Utilization (%)"}
    )
    
    for trace in temp_fig.data:
        trace.line.color = "royalblue"
        trace.showlegend = False 
        fig.add_trace(trace, row=i+1, col=1)


fig.update_layout(
    height=1200, 
    width=1400,
    title_text="Parking Utilization Rate Over Time",
    hovermode="x unified",
    template="plotly_white" 
)
fig.update_yaxes(range=[0, 1.1], tickformat=".0%")

fig.update_xaxes(title_text="Measured At", row=len(parking_lots), col=1)

fig.show()

Now utilisation rate ranges form 0 to 100% on all parkings 

In [ ]:
df_sks_users = df_sks_users.sort_values(by=['external_timestamp'])
df_sks_users = df_sks_users.reset_index(drop=True)
df_sks_users.head(10)

In [ ]:
check_missing_logs(df_sks_users, 'external_timestamp', 300, 120)

In [ ]:
df_sks_users.iloc[58322:58324]

In [ ]:
df_sks_users.iloc[5759:5761]

The second largest gap is in February, which is bad for using it to impute September (both are last months before the start of a new term). may prove insufficient, but data in February can be fixed using "healthy" 3 weeks up to February 22nd. 

In [ ]:
plt.figure(figsize=(50, 10))

sns.lineplot(data=df_sks_users, x="external_timestamp", y="active_users")

plt.show()

In [ ]:
def add_trend(df):
    epsilon = 1e-6
    shift_24h = df['spaces_left'].shift(288)
    df['trend_24h'] = (df['spaces_left'] - shift_24h) / (shift_24h + epsilon)
    return df

In [ ]:
df_parking_architektura_5min_clipped = add_trend(df_parking_architektura_5min_clipped)
df_parking_wronskiego_5min_clipped = add_trend(df_parking_wronskiego_5min_clipped)
df_parking_polinka_5min_clipped = add_trend(df_parking_polinka_5min_clipped)
df_parking_D20_5min_clipped = add_trend(df_parking_D20_5min_clipped)
df_parking_geocentrum_5min_clipped = add_trend(df_parking_geocentrum_5min_clipped)

In [ ]:
df_parking_architektura_5min_clipped.head()

In [ ]:
df_sks_users.head()

In [ ]:
df_sks_users['day'] = df_sks_users['external_timestamp'].dt.day_name()

df_sks_users[['external_timestamp', 'day']].tail()

In [ ]:
def is_open_sks(row):

    day = row['day']
    time = row['external_timestamp'].time()
    if day in ['Saturday', 'Sunday']:
        return False
    elif day == 'Friday':
        # check if between 7:30 and 16:00
        if dt.time(7, 30) <= time < dt.time(16, 00):
            return True
        else:
            return False
    else:
         # check if between 7:30 and 17:00
        if dt.time(7, 30) <= time < dt.time(17, 00):
            return True
        else:
            return False
    

In [ ]:
df_sks_users['is_open'] = df_sks_users.apply(is_open_sks, axis=1)

In [ ]:
agg_rules = {
    'active_users': 'mean',
    'moving_average_21': 'mean',
    'day': 'first', 
    'is_open': 'max',
}

df_sks_users_5min = df_sks_users.resample('5min', on='external_timestamp').agg(agg_rules)
df_sks_users_5min.head(20)

In [ ]:
neighbors = {
    'arch': df_parking_architektura_5min_clipped,
    'wron': df_parking_wronskiego_5min_clipped,
    'pol': df_parking_polinka_5min_clipped,
    'd20': df_parking_D20_5min_clipped,
    'geo': df_parking_geocentrum_5min_clipped
}

target_df = df_sks_users_5min.copy()
target_df = target_df.sort_index()

# Loop for merging neighbor features
for prefix, df_neighbor in neighbors.items():
    df_neighbor = df_neighbor.sort_index()

    cols_to_use = ['spaces_left', 'trend_24h']

    actual_cols = [c for c in cols_to_use if c in df_neighbor.columns]
    
    temp_df = df_neighbor[actual_cols].copy()
    
    #change column names
    rename_map = {col: f"{prefix}_{col}" for col in actual_cols}
    temp_df = temp_df.rename(columns=rename_map)

    target_df = pd.merge_asof(
        target_df,
        temp_df,
        left_index=True,   # Use external_timestamp (index) from target_df
        right_index=True,  # Use measured_at (index) from temp_df
        direction='nearest',
        tolerance=pd.Timedelta('5min')
    )

# Feature Engineering 
target_df['hour'] = target_df.index.hour
target_df['day_of_week'] = target_df.index.dayofweek
target_df['is_weekend'] = target_df['day_of_week'].isin([5, 6]).astype(int)

feature_cols = [c for c in target_df.columns if any(x in c for x in neighbors.keys())] 
feature_cols += ['hour', 'day_of_week', 'is_weekend']

target_col = 'active_users'


valid_features_mask = target_df[feature_cols].notna().all(axis=1)

train_mask = valid_features_mask & target_df[target_col].notna()

X_train = target_df.loc[train_mask, feature_cols]
y_train = target_df.loc[train_mask, target_col]

predict_mask = valid_features_mask & target_df[target_col].isna()
X_predict = target_df.loc[predict_mask, feature_cols]

print(f"Training on {len(X_train)} rows. Predictions for {len(X_predict)} rows.")

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    verbose=100,
    cat_features=['hour', 'day_of_week', 'is_weekend'] 
)

if not X_train.empty and not X_predict.empty:
    model.fit(X_train, y_train)

    # Imputation
    predicted_values = model.predict(X_predict)
    
    # Clip negative users
    predicted_values = np.clip(predicted_values, a_min=0, a_max=None)
    
    target_df.loc[predict_mask, target_col] = predicted_values
    
    # Flag
    target_df['is_imputed'] = 0
    target_df.loc[predict_mask, 'is_imputed'] = 1

    print("Imputation completed successfully.")
elif X_train.empty:
    print("Error: Empty training set. Check if date ranges overlap.")
else:
    print("No data to predict (no NaN in active_users with valid features).")
cols_to_keep = df_sks_users_5min.columns.tolist() 
if 'is_imputed' not in cols_to_keep:
    cols_to_keep.append('is_imputed')

df_sks_users_final = target_df[cols_to_keep].copy()

print(df_sks_users_final.head())

In [ ]:
plt.figure(figsize=(50, 10))

sns.lineplot(data=df_sks_users_final, x="external_timestamp", y="active_users", hue="is_imputed")

plt.show()

With imputing is done by CatBoostRegressor sks data looks better, but there still remains the question of its usability, as now it's based on trends from parking data, which it's supposed to help predict, so it kind of creates a feedback loop. Additionally, inputting itself doesn't seem to have been that much of a success, as predictions done by the model seem to be lower/higher than what was expected in some places. 

In [ ]:

class SksUSersInputer(BaseEstimator, TransformerMixin):
    def __init__(self, freq_minutes=5, semester_lag_weeks=30):
        self.freq_minutes = freq_minutes
        self.semester_lag_weeks = semester_lag_weeks # 30 weeks = approx. 7 months

    def fit(self, X, y=None):
        self.fill_value_ = X['active_users'].median()
        return self  

    def transform(self, X):
        X = X.copy()

        if not X.index.is_monotonic_increasing:
             X = X.sort_index()


        X['is_imputed'] = X['active_users'].isna().astype(int)
        
        rows_per_hour = 60 // self.freq_minutes
        rows_per_week = 7 * 24 * rows_per_hour
        rows_semester = rows_per_week * self.semester_lag_weeks

        cols_to_fix = X.select_dtypes(include=[np.number]).columns.tolist()
        
        for col in cols_to_fix:
            limit_small = rows_per_hour
            X[col] = X[col].interpolate(method='linear', limit=limit_small)

            mask_oct_missing = (X.index.month == 9) & (X[col].isna())
            
            semester_values = X[col].shift(rows_semester)
            X.loc[mask_oct_missing, col] = X.loc[mask_oct_missing, col].fillna(semester_values)

            for _ in range(3):
                X[col] = X[col].fillna(X[col].shift(rows_per_week))
            

            X[col] = X[col].ffill(limit=rows_per_hour*2).bfill(limit=rows_per_hour*2)

        
        X = X.ffill()
        X = X.fillna(self.fill_value_)
        
        return X
    
sks_users_inputer = SksUSersInputer()

In [ ]:
df_sks_users_5min_copy = df_sks_users_5min.copy()
df_sks_users_5min_copy = sks_users_inputer.fit_transform(df_sks_users_5min_copy)    

In [ ]:
plt.figure(figsize=(50, 10))

sns.lineplot(data=df_sks_users_5min_copy, x="external_timestamp", y="active_users", hue="is_imputed")

plt.show()

Using February to impute September seems to yield better results.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(50, 30), sharex=True)

sns.lineplot(data=df_sks_users_final, x="external_timestamp", y="active_users", hue="is_imputed", ax=axes[0])
axes[0].set_title("SKS users inputed - catboost model")

sns.lineplot(data=df_sks_users_5min_copy, x="external_timestamp", y="active_users", hue="is_imputed", ax=axes[1])
axes[1].set_title("SKS users inputed - lag semester")

sns.lineplot(data=df_parking_geocentrum_5min_clipped, x=df_parking_geocentrum_5min_clipped.index, y="utilization_rate", hue="is_imputed", ax=axes[2])
axes[1].set_title("Parking Geocentrum utilization rate")

plt.show()

<img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZDVhYnlnZms4ZzlyZjM2c20ydXRoM2tpNGZoMjdyYjBuYzVzMjNnMyZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3orif3Pcg3rLSwhwze/giphy.gif" width="400" height="300" />